In [1]:
import torch
import torch.nn as nn
torch.manual_seed(123)

In [2]:
class Example_Neural_Network(nn.Module):
  def __init__(self,layer_size,use_shortcut ):
    super().__init__()
    self.use_shortcut=use_shortcut
    self.layers=nn.ModuleList([
      nn.Sequential(nn.Linear(layer_size[0],layer_size[1]),nn.GELU()),      
      nn.Sequential(nn.Linear(layer_size[1],layer_size[2]),nn.GELU()),
      nn.Sequential(nn.Linear(layer_size[2],layer_size[3]),nn.GELU()),
      nn.Sequential(nn.Linear(layer_size[3],layer_size[4]),nn.GELU()),
      nn.Sequential(nn.Linear(layer_size[4],layer_size[5]),nn.GELU())
    ])
  def forward(self,x):
    for layer in self.layers:
      layer_out=layer(x)
      if(self.use_shortcut==True and  x.shape==layer_out.shape):
        x=x+layer_out
      else:
        x=layer_out  
    return x  
        

In [3]:
layer_sizes = [3,3,3,3,3,1]

sample_input = torch.tensor([[1., 0., 1,]])
torch.manual_seed(123)


In [4]:
def print_gradients(model, x):
    output = model(x) #ACTUAL OUTPUT
    target = torch.tensor([[0.]]) #EXPECTED OUTPUT
    
    loss = nn.MSELoss() #LOSS FN
    loss = loss(output, target) #LOSS
    
    loss.backward()
    
    grad = []
    for name, param in model.named_parameters():
        if 'weight' in name:
            g = param.grad.abs().mean().item()
            grad.append(g)
            print(f"{name} has gradient mean of {g}")
            
    return grad

In [5]:
model_without_shortcut = Example_Neural_Network(
    layer_sizes, use_shortcut=False)
model_without_shortcut

Example_Neural_Network(
  (layers): ModuleList(
    (0-3): 4 x Sequential(
      (0): Linear(in_features=3, out_features=3, bias=True)
      (1): GELU(approximate='none')
    )
    (4): Sequential(
      (0): Linear(in_features=3, out_features=1, bias=True)
      (1): GELU(approximate='none')
    )
  )
)

In [6]:
without_shortcut = print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.0001504371757619083
layers.1.0.weight has gradient mean of 0.00013967569975648075
layers.2.0.weight has gradient mean of 0.0006070034578442574
layers.3.0.weight has gradient mean of 0.0011253699194639921
layers.4.0.weight has gradient mean of 0.004502736497670412


In [7]:

model_with_shortcut = Example_Neural_Network(
    layer_sizes, use_shortcut=True)
model_with_shortcut


Example_Neural_Network(
  (layers): ModuleList(
    (0-3): 4 x Sequential(
      (0): Linear(in_features=3, out_features=3, bias=True)
      (1): GELU(approximate='none')
    )
    (4): Sequential(
      (0): Linear(in_features=3, out_features=1, bias=True)
      (1): GELU(approximate='none')
    )
  )
)

In [8]:
with_shortcut = print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.0026822444051504135
layers.1.0.weight has gradient mean of 0.00948988739401102
layers.2.0.weight has gradient mean of 0.008694369345903397
layers.3.0.weight has gradient mean of 0.012804403901100159
layers.4.0.weight has gradient mean of 0.07457051426172256


In [10]:
import pandas as pd 
pd.set_option("display.precision", 2)

df = pd.DataFrame({
    "without_shortcut": without_shortcut,
    "with_shortcut": with_shortcut,
    "diff%":(abs(df["with_shortcut"] - df["without_shortcut"])/df["without_shortcut"])*100
})
df

,without_shortcut,with_shortcut,diff%
0,1.50e-04,2.68e-03,1682.97
1,1.40e-04,9.49e-03,6694.23
2,6.07e-04,8.69e-03,1332.34
3,1.13e-03,1.28e-02,1037.80
4,4.50e-03,7.46e-02,1556.12
